# DSC0152 — notebook v71

## Tujuan

Mencari informasi baru yang tidak dimiliki keluarga v66–v70, bukan menambah
blok fitur redundan. Tiga hal diuji:

1. **historical panel** — 74.918 pinjaman lama dengan outcome 12 bulan lengkap;
2. **auxiliary rate** — residual rate terhadap fitur risiko;
3. **model diversity** — CatBoost + MLP di atas feature set v70.

Hasil akhirnya: panel dan rate residual **tidak** menambah model sekarang;
MLP adalah lever yang benar-benar baru. Notebook ini mencatat ketiganya,
termasuk hasil negatifnya.

## Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "agas").exists():
    ROOT = ROOT.parent
OUT = ROOT / "agas/v71/output"
V70 = ROOT / "agas/v70/output"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("root", ROOT)
print("v71 output", OUT)

root /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026
v71 output /Users/agas/Documents/ITB/Lomba/dsc-logika-ui-2026/agas/v71/output


## Public ledger

Anchor tetap `v66_04` = **0.56424**. v70 menguji tiga arah dan tidak ada yang
melewatinya: `v70_07` 0.56419 (praktis seri), `v70_06` 0.56040 (OOF KS
tertinggi tetapi public turun), `v70_01` 0.55801 (OOF AUC tertinggi tetapi
public paling rendah). Ini bukti bahwa selisih OOF di bawah ~0.001 tidak
memprediksi public.

In [2]:
ledger = pd.DataFrame([
    ("v66_04", "ext0.5|tm30|lgb_lean", 0.56424, "anchor"),
    ("v69_10", "ext0.5|tm30|extra", 0.55989, "pivot extra gagal"),
    ("v70_06", "plateau|tm30w7|v70_extra", 0.56040, "OOF KS tinggi, public turun"),
    ("v70_07", "ext0.5|tm30v70|lgb_lean", 0.56419, "praktis seri"),
    ("v70_01", "leg|v70_extra", 0.55801, "OOF AUC tinggi, public turun"),
], columns=["candidate", "chain", "public", "note"])
ledger

,candidate,chain,public,note
0,v66_04,ext0.5|tm30|lgb_lean,0.56424,anchor
1,v69_10,ext0.5|tm30|extra,0.55989,pivot extra gagal
2,v70_06,plateau|tm30w7|v70_extra,0.56040,"OOF KS tinggi, public turun"
3,v70_07,ext0.5|tm30v70|lgb_lean,0.56419,praktis seri
4,v70_01,leg|v70_extra,0.55801,"OOF AUC tinggi, public turun"


## 1. Historical panel

### Konstruksi label

`previous_applications` yang berstatus `approved` dan punya `loan_id` digabung
dengan repayment history. Label adalah `max_days_past_due >= 90` pada 12
angsuran pertama. Baris hanya dipakai jika window 12 bulan sudah selesai
sebelum `decision_datetime` aplikasi sekarang.

Label ini adalah **aproksimasi** dari definisi resmi “90+ DPD dalam 12 bulan”:
data repayment tidak memberi timestamp harian kapan DPD 90 tercapai, jadi
angsuran pertama sampai ke-12 dipakai sebagai proksi. Batas ini dicatat, bukan
disembunyikan.

### Fitur

Semua fitur dihitung pada `decision_datetime` pinjaman historis:

- terms pinjaman historis dan decision lag;
- previous applications sebelum cut: approval/rejection, amount, rate, recency,
  gap, active count;
- repayment sebelum cut: late30/late90, DPD, unpaid, delay, tren, outcome per
  loan;
- bureau report milik application historis: score, provider disagreement,
  utilization, inquiries, delinquency;
- account behavior sebelum cut: inflow/outflow, net, saldo, low-balance days,
  tren 1/3/6/12 bulan;
- e-commerce sebelum cut: spend/order 30/90/180/365 hari, basket, kategori,
  payment method, cancel, tren. Agregasi harian per customer dilakukan lebih
  dulu agar memori tetap wajar.

Profil customer adalah snapshot saat ini. Umur dan tenure di-backdate; income
dipakai dengan catatan bahwa ini snapshot, bukan nilai pada saat pinjaman
historis. Tidak ada target aplikasi sekarang yang dipakai.

In [3]:
build = json.loads((OUT / "summary_panel_build_v71.json").read_text())
panel = json.loads((OUT / "summary_panel_v71.json").read_text())
print(json.dumps(build, indent=2))
print("\npanel model:")
for k in ["hist_rows", "current_rows", "features", "hist_positive_rate",
          "panel_hist_auc", "panel_hist_ks", "panel_current_auc",
          "panel_current_ks", "minutes"]:
    print(f"  {k}: {panel[k]}")
print("\nper fold")
display(pd.DataFrame(panel["per_fold"]).T)

{
  "rows": 174918,
  "historical_rows": 74918,
  "current_rows": 100000,
  "historical_positive_rate": 0.16503377025547933,
  "features": 375,
  "minutes": 1.1594341158866883,
  "note": "All pn_ features use each row's own decision_datetime cutoff. Historical rows are approved loans with a fully observed 12-month window."
}

panel model:
  hist_rows: 74918
  current_rows: 100000
  features: 378
  hist_positive_rate: 0.16503377025547933
  panel_hist_auc: 0.7082830651075802
  panel_hist_ks: 0.29624027615940535
  panel_current_auc: 0.7561853829862684
  panel_current_ks: 0.3729769543334906
  minutes: 4.33166313568751

per fold


,auc,ks,n,pos
0,0.706473,0.298253,15095.0,0.165949
1,0.706688,0.300122,14982.0,0.162862
2,0.708302,0.300067,14986.0,0.164153
3,0.707914,0.296547,14946.0,0.165395
4,0.712004,0.303543,14909.0,0.166812


### Hasil panel dan keputusan

Panel model memakai 5-fold **GroupKFold by customer**. Historical OOF AUC
0.7083 dan KS 0.2962; ketika skor panel dipakai untuk memprediksi target
aplikasi sekarang, AUC-nya 0.7562. Itu bukan model utama, tetapi cukup untuk
diuji sebagai fitur.

Screening current model memakai 5 fold × 3 fold seed, config `extra`, factor
aktif. Pembanding adalah cache v70 yang sama.

In [4]:
rows = []
for name, path in [
    ("v70 baseline (no panel)", V70 / "summary_s4_all.json"),
    ("v71 panel score only", OUT / "summary_cv_v71_panel.json"),
    ("v71 panel score + 68 pn_ features + rf_", OUT / "summary_cv_v71_all.json"),
]:
    if path.exists():
        j = json.loads(path.read_text())
        rows.append({"run": name, "n_lean": j["n_lean"],
                     "mean_auc": j["mean_auc"], "sd_auc": j["sd_auc"],
                     "mean_ks": j["mean_ks"]})
pd.DataFrame(rows)

,run,n_lean,mean_auc,sd_auc,mean_ks
0,v70 baseline (no panel),684,0.847219,0.000183,0.527724
1,v71 panel score only,771,0.847202,0.000125,0.526964
2,v71 panel score + 68 pn_ features + rf_,850,0.847199,0.000130,0.526876


**Keputusan:** panel score hanya 0.847202 vs baseline 0.847219 (selisih
−0.000017, jauh di dalam noise). Panel + 68 fitur + rate residual 0.847199.
Panel **tidak dipakai** sebagai fitur current model. Ini negative result yang
jelas: label historisnya terlalu lemah/noisy untuk menambah model yang sudah
mempunyai 684 fitur current.

## 2. Auxiliary rate

Model multiclass memprediksi tier rate (16/20/24/28/32%) dari fitur selain
rate dan installment. Tujuannya mendapat residual rate terhadap grade risiko.
Hasilnya: rate bisa diprediksi hampir sempurna dari fitur lain (AUC rate>=24%
= 1.0), residual hampir nol (median absolut ~2.7e-6). Ini bukan sinyal baru;
ini berarti rate adalah fungsi deterministik dari profil yang sudah terukur.
Fitur `rf_*` tidak dipakai di model final.

In [5]:
bf = json.loads((OUT / "summary_build_features_v71.json").read_text())
print(json.dumps(bf["rate_model"], indent=2))
# residual distribution from the cache, if present
f = pd.read_pickle(OUT / "features_v71.pkl")
tr = f[f.target.notna()]
tr[["rf_rate_residual", "rf_rate_abs_residual", "rf_rate_entropy"]].describe(
    percentiles=[.01, .5, .99])

{
  "predictors": 861,
  "classes": [
    0.16,
    0.2,
    0.24,
    0.28,
    0.32
  ],
  "oof_auc_rate_ge_024": 1.0
}


,rf_rate_residual,rf_rate_abs_residual,rf_rate_entropy
count,70000.000000,7.000000e+04,70000.000000
mean,0.000015,3.318022e-05,0.004952
std,0.000561,5.601636e-04,0.024561
min,-0.002962,1.207617e-12,0.000101
1%,-0.000135,3.439956e-08,0.000211
50%,-0.000002,2.656058e-06,0.001345
99%,0.000119,2.766367e-04,0.057299
max,0.065609,6.560917e-02,0.904328


## 3. Model diversity — lever yang berhasil

Feature set v70 adalah 684 kolom lean; factor features tetap fold-safe.
LightGBM sudah jenuh. Yang baru adalah dua geometri model:

- **CatBoost**: split/regularisasi berbeda, 15 fold × 3 fold seed;
- **MLP**: 771 kolom numerik, median imputation + clipping + standardisasi
  di dalam fold, arsitektur 256→64, dropout, early stopping.

MLP tidak memakai panel/rate. CatBoost memakai lean yang sama dengan LGBM.

In [6]:
rows = []
for name, path in [
    ("MLP 256/64 (3 seeds × 3 fold seeds)", OUT / "summary_mlp_v71.json"),
    ("MLP wide 512/128 (15 fold, 3 fold seed)", OUT / "summary_mlp_wide_v71.json"),
    ("MLP wide 512/128 (screen 5-fold)", OUT / "summary_mlp_wide_screen.json"),
    ("CatBoost (screen 5-fold)", OUT / "summary_cat_v71_screen.json"),
    ("CatBoost (15 fold, 3 fold seed)", OUT / "summary_cat_v71.json"),
    ("XGBoost (screen 5-fold)", OUT / "summary_xgb_v71_screen.json"),
]:
    if path.exists():
        j = json.loads(path.read_text())
        rows.append({"leg": name, "mean_auc": j["mean_auc"],
                     "sd_auc": j["sd_auc"], "mean_ks": j["mean_ks"],
                     "minutes": j["minutes"]})
pd.DataFrame(rows)

,leg,mean_auc,sd_auc,mean_ks,minutes
0,MLP 256/64 (3 seeds × 3 fold seeds),0.848433,0.000145,0.529473,19.864212
1,"MLP wide 512/128 (15 fold, 3 fold seed)",0.846901,0.000325,0.526979,33.113701
2,MLP wide 512/128 (screen 5-fold),0.845859,0.000000,0.525550,2.767010
3,CatBoost (screen 5-fold),0.845663,0.000000,0.523017,3.732415
4,"CatBoost (15 fold, 3 fold seed)",0.846704,0.000109,0.526520,55.225634


MLP final (15 fold × 3 fold seed × 3 seed) mencapai AUC **0.848433** dan KS
**0.529473**, sedikit di atas `v70_extra` (0.848369 / 0.528503). Tetapi nilai
sebenarnya ada di korelasi: rank MLP vs v70 hanya ~0.973, jauh lebih rendah
daripada korelasi antar-LGBM. Blend rank 60% MLP + 40% v70 mencapai
**AUC 0.850082 / KS 0.531973** pada OOF.

CatBoost final dan blend final dibaca dari artefak di bawah.

In [7]:
# candidate table is the authoritative final blend/selection record
cand = json.loads((OUT / "summary_v71_candidates.json").read_text())
print("core weights:", {k: round(float(v), 3) for k, v in cand["core_weights"].items()})
print("core auc/ks:", round(cand["core_auc"], 6), round(cand["core_ks"], 6))
display(pd.DataFrame(cand["candidates"]))

core weights: {'mlp_mix': 0.6, 'mlp_base': 0.42, 'mlp_wide': 0.18, 'cat': 0.1, 'xgb': 0.0, 'v70': 0.3}
core auc/ks: 0.850147 0.532834


,slot,name,auc,ks,ks_minus_sigma,proxy,spearman_v66_04,file,valid
0,1,core71,0.850147,0.532834,0.529234,"{'mean_dks': 0.004244, 'sd_dks': 0.002434, 'p_...",0.986847,submission_v71_01_core71.csv,True
1,2,core71_ks,0.850112,0.532884,0.529284,"{'mean_dks': 0.004243, 'sd_dks': 0.002383, 'p_...",0.987708,submission_v71_02_core71_ks.csv,True
2,3,stack_lr,0.850049,0.531650,0.528050,"{'mean_dks': 0.003566, 'sd_dks': 0.002276, 'p_...",0.988671,submission_v71_03_stack_lr.csv,True
3,4,anchor0.50|mlp_base,0.849669,0.530838,0.527238,"{'mean_dks': 0.00285, 'sd_dks': 0.00187, 'p_ks...",0.994196,submission_v71_04_anchor0.50_mlp_base.csv,True
4,5,tm30_71,0.849565,0.530865,0.527265,"{'mean_dks': 0.00294, 'sd_dks': 0.001842, 'p_k...",0.994328,submission_v71_05_tm30_71.csv,True
5,6,ext0.25|tm30_71|lgb_lean,0.849470,0.531136,0.527536,"{'mean_dks': 0.003211, 'sd_dks': 0.002061, 'p_...",0.991736,submission_v71_06_ext0.25_tm30_71_lgb_lean.csv,True
6,7,anchor0.50|core71,0.849395,0.530802,0.527202,"{'mean_dks': 0.002587, 'sd_dks': 0.001581, 'p_...",0.996729,submission_v71_07_anchor0.50_core71.csv,True
7,8,leg|mlp_base,0.849274,0.530920,0.527320,"{'mean_dks': 0.003059, 'sd_dks': 0.002792, 'p_...",0.976866,submission_v71_08_leg_mlp_base.csv,True
8,9,ext0.5|tm30_71|lgb_lean,0.849043,0.530925,0.527325,"{'mean_dks': 0.002854, 'sd_dks': 0.002287, 'p_...",0.987995,submission_v71_09_ext0.5_tm30_71_lgb_lean.csv,True
9,10,leg|mlp_wide,0.848776,0.530439,0.526839,"{'mean_dks': 0.002062, 'sd_dks': 0.002864, 'p_...",0.976296,submission_v71_10_leg_mlp_wide.csv,True


### Public-proxy bootstrap

Untuk setiap kandidat, 600 subset stratified berukuran 30.000 diambil dari OOF,
lalu dihitung `KS(candidate) - KS(anchor)` pada subset yang sama. Ini bukan
prediksi public; ini mengukur apakah kandidat menang karena perbedaan rank yang
konsisten, bukan karena satu argmax KS.

Yang dicari adalah kandidat dengan `P(KS lebih baik) > 0.55` dan AUC yang masih
masuk lantai 0.0020 di bawah kandidat terbaik. Kandidat yang rank-nya >0.9995
sama dengan kandidat yang sudah dipilih dideduplikasi.

In [8]:
rows = []
for r in cand["candidates"]:
    rows.append({
        "slot": r["slot"], "name": r["name"], "auc": r["auc"], "ks": r["ks"],
        "dKS": r["proxy"]["mean_dks"], "P(ks)": r["proxy"]["p_ks_better"],
        "dAUC": r["proxy"]["mean_dauc"], "rho_v66": r["spearman_v66_04"],
        "file": r["file"], "valid": r["valid"],
    })
pd.DataFrame(rows)

,slot,name,auc,ks,dKS,P(ks),dAUC,rho_v66,file,valid
0,1,core71,0.850147,0.532834,0.004244,0.965000,0.003330,0.986847,submission_v71_01_core71.csv,True
1,2,core71_ks,0.850112,0.532884,0.004243,0.966667,0.003292,0.987708,submission_v71_02_core71_ks.csv,True
2,3,stack_lr,0.850049,0.531650,0.003566,0.951667,0.003230,0.988671,submission_v71_03_stack_lr.csv,True
3,4,anchor0.50|mlp_base,0.849669,0.530838,0.002850,0.945000,0.002846,0.994196,submission_v71_04_anchor0.50_mlp_base.csv,True
4,5,tm30_71,0.849565,0.530865,0.002940,0.943333,0.002741,0.994328,submission_v71_05_tm30_71.csv,True
5,6,ext0.25|tm30_71|lgb_lean,0.849470,0.531136,0.003211,0.946667,0.002650,0.991736,submission_v71_06_ext0.25_tm30_71_lgb_lean.csv,True
6,7,anchor0.50|core71,0.849395,0.530802,0.002587,0.961667,0.002566,0.996729,submission_v71_07_anchor0.50_core71.csv,True
7,8,leg|mlp_base,0.849274,0.530920,0.003059,0.873333,0.002472,0.976866,submission_v71_08_leg_mlp_base.csv,True
8,9,ext0.5|tm30_71|lgb_lean,0.849043,0.530925,0.002854,0.895000,0.002226,0.987995,submission_v71_09_ext0.5_tm30_71_lgb_lean.csv,True
9,10,leg|mlp_wide,0.848776,0.530439,0.002062,0.750000,0.001977,0.976296,submission_v71_10_leg_mlp_wide.csv,True


## Validasi CSV

Semua file harus 30.000 baris, `application_id` unik dan cocok dengan
`data/test_applications.csv`, probabilitas di [0, 1], dan tanpa NaN.

In [9]:
test_ids = pd.read_csv(ROOT / "data/test_applications.csv").application_id
bad = []
for p in sorted(OUT.glob("submission_v71_*.csv")):
    d = pd.read_csv(p)
    ok = (len(d) == 30000 and d.application_id.is_unique
          and set(d.application_id) == set(test_ids)
          and d.default_probability.notna().all()
          and d.default_probability.between(0, 1).all())
    if not ok:
        bad.append(p.name)
    print(f"{p.name:60s} rows={len(d):5d} ok={ok}")
print("bad:", bad)

submission_v71_01_core71.csv                                 rows=30000 ok=True
submission_v71_02_core71_ks.csv                              rows=30000 ok=True
submission_v71_03_stack_lr.csv                               rows=30000 ok=True
submission_v71_04_anchor0.50_mlp_base.csv                    rows=30000 ok=True
submission_v71_05_tm30_71.csv                                rows=30000 ok=True
submission_v71_06_ext0.25_tm30_71_lgb_lean.csv               rows=30000 ok=True
submission_v71_07_anchor0.50_core71.csv                      rows=30000 ok=True
submission_v71_08_leg_mlp_base.csv                           rows=30000 ok=True
submission_v71_09_ext0.5_tm30_71_lgb_lean.csv                rows=30000 ok=True


submission_v71_10_leg_mlp_wide.csv                           rows=30000 ok=True
submission_v71_11_anchor0.25_core71.csv                      rows=30000 ok=True
submission_v71_12_ext0.75_tm30_71_lgb_lean.csv               rows=30000 ok=True


submission_v71_13_leg_v70_avg.csv                            rows=30000 ok=True
submission_v71_14_anchor_v66_04.csv                          rows=30000 ok=True
bad: []


## Reproduksi

Jalankan dari root repo:

```bash
python3 agas/v71/training/build_panel_v71.py
python3 agas/v71/training/train_panel_v71.py
python3 agas/v71/training/build_features_v71.py
python3 agas/v71/training/train_mlp_v71.py --drops pn_,rf_ --folds 15 \
  --fold-seeds 42,7,13 --seeds 0,1,2 --out-prefix mlp_v71
python3 agas/v71/training/train_cat_v71.py --drops pn_,rf_ --folds 15 \
  --fold-seeds 42,7,13 --seeds 0 --use-factor --out-prefix cat_v71
python3 agas/v71/training/train_mlp_v71.py --drops pn_,rf_ --folds 15 \
  --fold-seeds 42,7,13 --seeds 0 --hidden 512 --hidden2 128 \
  --dropout 0.35 --dropout2 0.20 --lr 0.001 --wd 0.0003 \
  --out-prefix mlp_wide_v71
python3 agas/v71/training/make_submission_v71.py
```

Notebook ini tidak menjalankan training. Set `RUN_TRAINING=True` hanya jika
ingin mencetak ulang perintah di atas.

In [10]:
RUN_TRAINING = False
if RUN_TRAINING:
    print("See the markdown cell above for the full command list.")
else:
    print("report-only mode; all numbers read from agas/v71/output")

report-only mode; all numbers read from agas/v71/output


## Kesimpulan

- **Panel historical** adalah negative result yang mahal tapi jelas: label
  historis 16.5% positif, model panel AUC 0.708, dan tidak menambah model
  current.
- **Rate residual** juga negative: rate deterministik dari fitur yang sudah ada.
- **MLP** adalah satu-satunya lever baru yang berhasil. OOF blend naik dari
  0.84837 (LGBM-only) ke ~0.8501, dengan KS ~0.5320.
- Public 0.577 tetap tidak dijamin. Kenaikan OOF ~0.0017 setara ~0.3 sd public;
  kandidat v71 lebih kuat dan lebih beragam, tetapi 0.577 masih membutuhkan
  favorable public variance atau structural break berikutnya.